# 00 — Experiment Plan: Options Flow as a Directional Signal for SPY

**Series:** Piccolo ML Options Strategy Research  
**Date:** 2026-02-20  
**Status:** Complete


## 1. Research Question

> **Can options market microstructure data — specifically Gamma Exposure (GEX),
> put/call ratios, implied-volatility skew, and open-interest concentration —
> reliably predict the *short-term directional bias* of SPY on a 5–10 day
> forward horizon, well enough to generate risk-adjusted alpha net of
> transaction costs?**

This question sits at the intersection of market microstructure, derivatives
theory, and applied machine learning. Options dealers must delta-hedge their
books; that hedging flow creates measurable, non-trivial price impacts.  
We hypothesise that aggregating these signals via an ensemble of walk-forward
XGBoost classifiers can produce a statistically significant edge.


## 2. Hypotheses

| ID  | Hypothesis | Mechanism |
|-----|-----------|----------|
| H1  | High net GEX suppresses realised volatility and biases SPY toward mean-reversion | Dealer delta-hedging acts as a vol-dampener |
| H2  | Elevated put/call OI ratio predicts downward pressure | Retail/institutional demand for puts signals bearish positioning |
| H3  | Negative IV skew (calls > puts IV) signals upside momentum | Risk reversal demand reflects directional bias |
| H4  | Combining H1–H3 with price-regime filters (SMA-200, vol regime) improves Sharpe | Regime conditioning reduces noise |
| H5  | A rolling walk-forward ensemble outperforms a static model | Markets are non-stationary; recency weighting matters |


## 3. Methodology Overview

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        PICCOLO PIPELINE                                 │
│                                                                         │
│  RAW DATA                FEATURE STORE           MODEL LAYER            │
│  ──────────              ─────────────           ───────────            │
│  IBKR EOD prices  ──►   price features     ──►  Walk-forward           │
│  IBKR options     ──►   options features   ──►  XGBoost folds          │
│  chain snapshots         Greeks-based            ↓                      │
│  CBOE historical  ──►   vol/regime         ──►  Exponential            │
│  data                    features               ensemble               │
│                                ↓                       ↓               │
│                         DuckDB (HIST +        Confidence-gated         │
│                         LIVE DBs)             signals → backtest       │
└─────────────────────────────────────────────────────────────────────────┘
```

**Label construction:** 3-class (Up=+1, Flat=0, Down=-1) using path-forward
returns over `LABEL_HORIZON_DAYS` days.  A return above `UP_THRESHOLD` → Up;
below `DOWN_THRESHOLD` → Down; otherwise Flat.

**Training:** Rolling walk-forward validation.  Each fold trains on
`N_TRAIN_MONTHS` months, tests on the following `N_TEST_MONTHS` months.
Folds are combined with exponential recency weighting (`ALPHA=0.7`).

**Signal gating:** Predictions are only acted on when the predicted-class
probability exceeds `CONF_THRESHOLD_UP` or `CONF_THRESHOLD_DOWN` AND the
regime filter (SMA-200, vol regime) passes.


## 4. Data Sources

| Source | Content | Cadence | Module |
|--------|---------|---------|--------|
| Theta Data terminal | EOD OHLCV for SPY (and universe) | Daily | `eod_prices_td.py` |
| Theta Data terminal | Options chain (OI, IV, GEX, put/call ratios) | Daily | `td_options_snapshot.py` |
| IBKR (legacy) | EOD prices and options chain | Daily | `legacy/bootstrap_eod_prices_ibkr.py`, `legacy/eod_prices_daily_ibkr.py`, `legacy/ibkr_options_snapshot.py` |
| CBOE historical | Long-run options data (14 years) | Static | CBOE data pipeline |
| DuckDB (HIST) | Persistent feature store (historical) | Updated daily | `config/settings.py` → `DUCKDB_PATH_HIST` |
| DuckDB (LIVE) | Intraday / live signals | Real-time | `config/settings.py` → `DUCKDB_PATH_LIVE` |


## 5. Success Criteria

| Metric | Target | Notes |
|--------|--------|-------|
| Out-of-sample accuracy | > 52% (3-class adjusted) | Better than random on directional calls |
| Sharpe ratio (signal-only) | > 0.8 annualised | Frictionless backtest (see Notebook 04 for cost sensitivity) |
| Max drawdown | < 25% | On equity curve |
| Win rate (Up/Down signals) | > 55% | Exclude Flat signals from denominator |
| Profit factor | > 1.3 | Gross profit / gross loss |


## 6. Notebook Series Map

| Notebook | Topic |
|----------|-------|
| **00_experiment_plan** (this) | Research question, hypotheses, overview |
| [01_data_pipeline](01_data_pipeline.ipynb) | Data ingestion, DuckDB schema, quality checks |
| [02_feature_engineering](02_feature_engineering.ipynb) | Feature construction, distributions, label building |
| [03_model_training_walkforward](03_model_training_walkforward.ipynb) | Walk-forward XGBoost, ensemble, confidence tuning |
| [04_backtest_performance](04_backtest_performance.ipynb) | Equity curve, Sharpe, drawdown, regime impact |


## 7. Configuration Reference

All key hyperparameters live in `src/piccolo/config_strategy.py`.  
Refer to them by name throughout the notebooks — never hardcode values.

Key parameters to note:
- `UP_THRESHOLD`, `DOWN_THRESHOLD` — label boundaries
- `LABEL_HORIZON_DAYS` — forward return horizon for label construction
- `N_TRAIN_MONTHS`, `N_TEST_MONTHS` — walk-forward window sizes
- `ALPHA` — exponential ensemble recency weight
- `CONF_THRESHOLD_UP`, `CONF_THRESHOLD_DOWN` — signal confidence gates
- `DUCKDB_PATH_HIST`, `DUCKDB_PATH_LIVE` — database paths (set in .env)


In [1]:
# Quick sanity-check: confirm the piccolo package is importable
import sys, os

# Adjust path so repo root is on sys.path
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

try:
    import src.piccolo.config_strategy as cfg
    print("config_strategy imported successfully")
    print(f"  UP_THRESHOLD       = {cfg.UP_THRESHOLD}")
    print(f"  DOWN_THRESHOLD     = {cfg.DOWN_THRESHOLD}")
    print(f"  LABEL_HORIZON_DAYS = {cfg.LABEL_HORIZON_DAYS}")
    print(f"  N_TRAIN_MONTHS     = {cfg.N_TRAIN_MONTHS}")
    print(f"  N_TEST_MONTHS      = {cfg.N_TEST_MONTHS}")
    print(f"  ALPHA              = {cfg.ALPHA}")
except ImportError as e:
    print(f"Import failed: {e}")
    print("Make sure you are running with the piccolo-public root on sys.path")


config_strategy imported successfully
  UP_THRESHOLD       = 0.015
  DOWN_THRESHOLD     = -0.015
  LABEL_HORIZON_DAYS = 5
  N_TRAIN_MONTHS     = 36
  N_TEST_MONTHS      = 3
  ALPHA              = 0.7


## 8. Next Steps

1. Run `01_data_pipeline.ipynb` to validate data availability and quality.
2. Proceed in notebook order.
3. Document results and anomalies in each notebook's *Findings* section.
4. After completing all notebooks, update the **Results Summary** in the root
   `README.md`.
